In [ ]:
!pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl triton cut_cross_entropy unsloth_zoo
!pip install sentencepiece protobuf datasets huggingface_hub hf_transfer
!pip install --no-deps unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "parquet",
    data_files={
        "train": "/kaggle/input/afrihealth-amharic/train.parquet",
        "validation": "/kaggle/input/afrihealth-amharic/validation.parquet",
    }
)

amharic_train = dataset["train"].filter(lambda x: x["language_country"] == "Amh_Eth")
amharic_val = dataset["validation"].filter(lambda x: x["language_country"] == "Amh_Eth")

print(f"Amharic train: {len(amharic_train)}, val: {len(amharic_val)}")

In [ ]:
def format_prompt(example):
    text = f"""### Instruction:
እባክህ ለሚከተለው የጤና ጥያቄ መልስ ስጥ።

### Question:
{example['question']}

### Answer:
{example['answer']}"""
    return {"text": text}

amharic_train = amharic_train.map(format_prompt)
amharic_val = amharic_val.map(format_prompt)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = amharic_train,
    eval_dataset = amharic_val,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 2,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

In [ ]:
model.save_pretrained("/kaggle/working/qwen_amharic_adapter")
tokenizer.save_pretrained("/kaggle/working/qwen_amharic_adapter")
print("Adapter saved.")